<a href="https://colab.research.google.com/github/Michaeltje26/AI-Cover-Maker-V1-V2/blob/main/assets/RVCAICoverMakerUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **RVC AI Cover Maker UI**
- Created by [Shirou](https://github.com/ShiromiyaG)
- Maintained by [Not Eddy (Spanish Mod)](http://discord.com/users/274566299349155851) in [AI HUB](https://discord.gg/aihub) community
- Improved the Start UI Cell with other Tunnel types by [Nick088](https://linktr.ee/Nick088)
<br>This colab uses the following projects:
- [Music Source Separation Universal Training Code](https://github.com/ZFTurbo/Music-Source-Separation-Training) by [ZFTurbo](https://github.com/ZFTurbo)
- [Applio](https://github.com/IAHispano/Applio) by [IAHispano](https://github.com/IAHispano)

In [2]:
#@title ## **Install**

!sudo apt update
!sudo apt install python3.10
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!sudo update-alternatives --set python3 /usr/bin/python3.10
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3
import sys
sys.path.append('/usr/local/lib/python3.10/dist-packages')

import os
import codecs
from IPython.display import clear_output
print("Installing requirements")
repo = "https://github.com/Michaeltje26/AI-Cover-Maker-V1-V2.git"
!git clone $repo main_program
%cd main_program
!pip install uv pyngrok -q
!uv pip install --no-deps -r requirements.txt -q
!uv pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --upgrade --index-url https://download.pytorch.org/whl/cu121 -q
!uv pip install numpy==1.23.5 -q
!python programs/applio_code/rvc/lib/tools/prerequisites_download.py
clear_output()
print("Requirements installed!")

Requirements installed!


In [3]:
%%bash
cd /content/main_program
git checkout -- programs/music_separation_code/utils.py

In [4]:
%%bash
python -m py_compile /content/main_program/programs/music_separation_code/utils.py

In [5]:
!sed -n '1,60p' /content/main_program/models/bs_roformer/mel_band_roformer.py

sed: can't read /content/main_program/models/bs_roformer/mel_band_roformer.py: No such file or directory


In [6]:
%%bash
fuser -k 7755/tcp || true

In [7]:
%%bash
FILE="/content/main_program/programs/music_separation_code/models/bs_roformer/mel_band_roformer.py"

# laat even zien of we de beartype import hebben
grep -n "from beartype import beartype" "$FILE" || true

# voeg direct na die import de disable-regel toe (maar alleen als hij er nog niet staat)
python - <<'PY'
from pathlib import Path
p = Path("/content/main_program/programs/music_separation_code/models/bs_roformer/mel_band_roformer.py")
txt = p.read_text(encoding="utf-8")

if "beartype = lambda f: f" in txt:
    print("✅ beartype was al disabled")
else:
    txt2 = txt.replace("from beartype import beartype", "from beartype import beartype\nbeartype = lambda f: f", 1)
    if txt2 == txt:
        raise SystemExit("Kon 'from beartype import beartype' niet vinden in het bestand.")
    p.write_text(txt2, encoding="utf-8")
    print("✅ beartype disabled in mel_band_roformer.py")
PY

11:from beartype import beartype
✅ beartype disabled in mel_band_roformer.py


In [1]:
import os, subprocess, torch
print("CUDA available:", torch.cuda.is_available())
print("torch cuda version:", torch.version.cuda)
print(subprocess.getoutput("ls -la /dev/nvidia* 2>/dev/null || echo 'no /dev/nvidia*'"))
print(subprocess.getoutput("nvidia-smi 2>/dev/null | head -n 20 || echo 'nvidia-smi not available'"))

CUDA available: True
torch cuda version: 12.8
crw-rw-rw- 1 root root 195,   0 Mar  1 18:52 /dev/nvidia0
crw-rw-rw- 1 root root 195, 255 Mar  1 18:52 /dev/nvidiactl
crw-rw-rw- 1 root root 241,   0 Mar  1 18:52 /dev/nvidia-uvm
crw-rw-rw- 1 root root 241,   1 Mar  1 18:52 /dev/nvidia-uvm-tools

/dev/nvidia-caps:
total 0
drwxr-xr-x 2 root root     80 Mar  1 18:52 .
drwxr-xr-x 6 root root    460 Mar  1 18:52 ..
cr-------- 1 root root 244, 1 Mar  1 18:52 nvidia-cap1
cr--r--r-- 1 root root 244, 2 Mar  1 18:52 nvidia-cap2
Sun Mar  1 18:57:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  C

In [ ]:
#@title ## **Start UI (fixed)**

#@markdown The type of tunnel you wanna use for seeing the public link, so that if one of them is down, you can use the other one.
Tunnel = "Gradio" #@param ["Gradio", "Ngrok", "LocalTunnel", "Horizon"]

#@markdown Also when using Ngrok, LocalTunnel or Horizon, you need to wait for the Local URL to appear, and only after that click on the Public URL which is above.

#@markdown Use the following option **only if you chose Ngrok** as the Tunnel:
#@markdown You can get the Ngrok Authtoken here: https://dashboard.ngrok.com/tunnels/authtokens/new.
ngrok_authtoken = "" #@param {type:"string"}

# @markdown You can optionally change the Ngrok Tunnel Region to one nearer to you for lower latency
ngrok_region = "us - United States (Ohio)" # @param ["au - Australia (Sydney)","eu - Europe (Frankfurt)", "ap - Asia/Pacific (Singapore)", "us - United States (Ohio)", "jp - Japan (Tokyo)", "in - India (Mumbai)","sa - South America (Sao Paulo)"]

#@markdown Use the following option **only if you chose Horizon** as the Tunnel:
#@markdown You can get the Horizon ID here: https://hrzn.run/dashboard/
horizon_id = "" #@param {type:"string"}


import os
import codecs
import glob
from IPython.display import clear_output

# --- Ensure we are in the right folder ---
BASE_DIR = "/content/main_program"
if os.path.isdir(BASE_DIR):
    os.chdir(BASE_DIR)
else:
    # fallback: try to find a folder that contains main.py
    candidates = glob.glob("/content/**/main.py", recursive=True)
    if not candidates:
        raise FileNotFoundError("Could not find main.py anywhere under /content. Did the repo clone step run correctly?")
    BASE_DIR = os.path.dirname(candidates[0])
    os.chdir(BASE_DIR)

# main.py (rot13 kept, but now we run it from the correct folder)
RUNTIME = codecs.decode("znva.cl", "rot_13")  # "main.py"

# --- Tunnel setup ---
if Tunnel == "Gradio":
    share_flag = "--share"
elif Tunnel == "Ngrok":
    share_flag = ""
    from pyngrok import conf, ngrok
    NgrokConfig = conf.PyngrokConfig()
    NgrokConfig.auth_token = ngrok_authtoken
    NgrokConfig.region = ngrok_region[0:2]
    conf.set_default(NgrokConfig)
    main_tunnel = ngrok.connect(7755)
    print("Ngrok Tunnel Public URL:", main_tunnel.public_url)
elif Tunnel == "LocalTunnel":
    share_flag = ""
    !npm install -g localtunnel
    import time
    import urllib.request

    with open('url.txt', 'w') as file:
        file.write('')

    get_ipython().system_raw('lt --port 7755 >> url.txt 2>&1 &')
    time.sleep(4)

    endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()

    with open('url.txt', 'r') as file:
        tunnel_url = file.read().replace("your url is: ", "").strip()

    clear_output()
    print(f"LocalTunnel Tunnel Public URL: \033[0m\033[93m{tunnel_url}\033[0m", end="\033[0m")
    print(f'LocalTunnel Password: {endpoint_ip}')
elif Tunnel == "Horizon":
    share_flag = ""
    !npm install -g @hrzn/cli
    import time

    !hrzn login $horizon_id

    with open('url.txt', 'w') as file:
        file.write('')

    get_ipython().system_raw('hrzn tunnel http://localhost:7755 >> url.txt 2>&1 &')
    time.sleep(4)

    tunnel_url = !grep -oE "https://[a-zA-Z0-9.-]+\.hrzn\.run" url.txt
    tunnel_url = tunnel_url[0] if len(tunnel_url) else ""

    clear_output()
    print(f"Horizon Tunnel Public URL: \033[0m\033[93m{tunnel_url}\033[0m")

# --- Kill previously running processes on port 7755 (don't fail if none) ---
!fuser -k 7755/tcp || true

# --- Start UI ---
print("Working directory:", os.getcwd())
print("Launching:", os.path.join(os.getcwd(), RUNTIME))

command = f"python {RUNTIME} {share_flag}"
!{command}

Working directory: /content/main_program
Launching: /content/main_program/main.py
/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.10/dist-packages/transformers/utils/generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
theme_schema%400.0.1.json: 13.5kB [00:00, 46.7MB/s]
Running on local URL:  http://127.0.0.1:7755
IMPORTANT: You are using gradio version 4.36.0, however version 4.44.1 is available, p